# Domain-Specific Fine-Tuning (Colab Optimized)

This notebook fine-tunes a Transformer encoder on the San Diego Municipal Code using a Masked Language Modeling (MLM) objective for domain adaptation.

**This notebook is recommended to run on Google Colab with a GPU (T4 or higher).**

### Objectives
1. **Tokenize the Legal Corpus**: Process the ~52k document chunks for model training.
2. **Masked Language Modeling (MLM)**: Perform domain adaptation for legal vocabulary.
3. **Model Export**: Save and package the fine-tuned weights for local RAG integration.

## 1. Environment Setup & Drive Mount

Initialize the training environment by installing necessary libraries and mounting Google Drive. This section handles the conditional setup for both Colab and local environments, ensuring that domain adaptation can proceed with access to large-scale storage for model weights.

In [ ]:
# Install training libraries
!pip install transformers datasets accelerate -q

import torch
import os
import math
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if IN_COLAB:
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("Running locally. Drive mount skipped.")


## 2. Dataset Processing

Load the pre-processed San Diego Municipal Code chunks and prepare them for Masked Language Modeling. This stage involves initializing a legal-specific tokenizer and mapping it across the corpus to create the tensor inputs required for Transformer training.

In [ ]:
# 1. Load Dataset
dataset = load_dataset("json", data_files=dataset_filename, split="train")

# 2. Initialize Tokenizer (from Legal-BERT)
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Tokenization Function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    num_proc=4, 
    remove_columns=dataset.column_names
)

print(f"Total tokenized records: {len(tokenized_dataset)}")

## 3. MLM Training

Execute the Masked Language Modeling (MLM) objective. By masking a percentage of tokens and tasking the model with their reconstruction, we adapt the underlying BERT encoder to the specific linguistic patterns and legal citations unique to the San Diego Municipal Code.

In [ ]:
# 1. Split the dataset for training and evaluation (90/10 split)
print("Splitting dataset into train and validation sets...")
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# 2. Initialize Model
model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)

# 3. Setup Data Collator (15% random masking)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

# 4. Define Training Arguments
training_args = TrainingArguments(
    output_dir="sd_legal_bert_mlm",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    push_to_hub=False
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Starting training loop...")
trainer.train()


## 4. Evaluation and Perplexity

Quantify the success of the domain adaptation by calculating the model's loss and perplexity on a held-out validation set. Perplexity measures how well the probability distribution predicted by the model matches the actual distribution of the text; a lower score indicates a more 'familiar' model.

In [ ]:

print("Evaluating model performance...")
eval_results = trainer.evaluate()

loss = eval_results["eval_loss"]
perplexity = math.exp(loss)

print(f"\n--- Final Metrics ---")
print(f"Evaluation Loss: {loss:.4f}")
print(f"Perplexity:      {perplexity:.4f}")

## 5. Model Export

Persist the fine-tuned model weights and tokenizer configuration to permanent storage. Saving directly to Google Drive ensures the trained artifacts are preserved and can be easily transferred back to the local `models/` directory for RAG integration.

In [ ]:
# 1. Save directly to Google Drive
save_dir = "/content/drive/MyDrive/sd-land-use-rag/models/san_diego_legal_bert"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model successfully saved to {save_dir}!")